# 137 · DeerFlow Runtime Workbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/main/examples/137-deerflow-runtime-workbook/deerflow_runtime_workbook.ipynb)

**What this notebook does:** Connect to a *running* DeerFlow instance, upload a markdown corpus, stream an agent run, and download the generated artifact — all from a Jupyter cell.

**Pattern:** Notebook-as-operator-console — the notebook drives an external runtime, not a self-contained script.

**Prerequisites:**
- DeerFlow running locally (`make dev` or `make docker-start` in the deer-flow repo)
- `OPENAI_API_KEY` set in your environment or `.env`

| Cell | Purpose |
|------|---------|
| 1 | Install + imports |
| 2 | Config + connection check (fails fast if DeerFlow is down) |
| 3 | Upload corpus files |
| 4 | Stream an agent run and render events live |
| 5 | List artifacts and preview the generated output |

In [ ]:
# Cell 1 — Install dependencies
%pip install -q httpx python-dotenv

In [ ]:
import json
import os
from IPython.display import display, Markdown

import httpx
from dotenv import load_dotenv

load_dotenv()
print('imports ok')

In [ ]:
# Cell 2 — Config + connection check
# This cell fails fast with a clear message if DeerFlow is not running.

BASE_URL = os.getenv('DEERFLOW_BASE_URL', 'http://localhost:8000')
THREAD_ID = 'demo-thread-137'

try:
    r = httpx.get(f'{BASE_URL}/api/health', timeout=5)
    r.raise_for_status()
    print(f'✓ DeerFlow is reachable at {BASE_URL}')
    print(f'  status: {r.json()}')
except Exception as exc:
    raise RuntimeError(
        f'DeerFlow is not reachable at {BASE_URL}.\n'
        'Run `make dev` (or `make docker-start`) in the deer-flow repo first.'
    ) from exc

In [ ]:
# Cell 3 — Upload corpus files

CORPUS = {
    '01-state-graph.md': 'State Graph: explicit directed graph. Nodes process state; edges route flow.',
    '02-reactive-loop.md': 'Reactive Loop: agent calls tools in a loop until a stop condition is met.',
    '03-multi-agent.md': 'Multi-Agent: supervisor delegates sub-tasks to specialist worker agents.',
}

http = httpx.Client(timeout=httpx.Timeout(30.0, read=120.0))
artifact_ids = {}

for filename, content in CORPUS.items():
    resp = http.post(
        f'{BASE_URL}/api/files/upload',
        files={'file': (filename, content.encode(), 'text/markdown')},
        data={'thread_id': THREAD_ID},
    )
    resp.raise_for_status()
    artifact_ids[filename] = resp.json().get('artifact_id', 'unknown')
    print(f'  {filename} → {artifact_ids[filename]}')

print(f'\n✓ Uploaded {len(artifact_ids)} files to thread {THREAD_ID!r}')

In [ ]:
# Cell 4 — Stream an agent run and render events live

TASK = (
    'Summarise the uploaded notes into a structured comparison table: '
    'pattern name, best use case, and main trade-off. '
    'Keep it under 200 words.'
)

events: list[tuple[str, dict]] = []
chunks: list[str] = []

print('Streaming…\n')
with http.stream(
    'POST',
    f'{BASE_URL}/api/chat/stream',
    json={'message': TASK, 'thread_id': THREAD_ID, 'plan_mode': False},
) as r:
    r.raise_for_status()
    for line in r.iter_lines():
        if not line.startswith('data:'):
            continue
        raw = line.removeprefix('data:').strip()
        if raw == '[DONE]':
            events.append(('end', {}))
            break
        try:
            ev = json.loads(raw)
            et = ev.get('type', 'unknown')
            events.append((et, ev))
            if et == 'message_chunk':
                chunk = ev.get('content', '')
                chunks.append(chunk)
                print(chunk, end='', flush=True)
        except json.JSONDecodeError:
            pass

print('\n')
counts: dict[str, int] = {}
for et, _ in events:
    counts[et] = counts.get(et, 0) + 1
print(f'Event counts: {counts}')

In [ ]:
# Cell 5 — Render output and compare runtime approaches

from IPython.display import display, Markdown

full_response = ''.join(chunks)
print('=== Agent Output ===')
display(Markdown(full_response))

print('\n=== When to use each DeerFlow approach ===')
print("""
┌────────────────────────┬─────────────────────────────────┬──────────────────────────────┐
│ Approach               │ When to use                     │ What the client does         │
├────────────────────────┼─────────────────────────────────┼──────────────────────────────┤
│ Embedded client (134)  │ Embed DeerFlow inside your app  │ Imports DeerFlowClient SDK   │
│ Custom skill (135)     │ Reuse a prompt-module pattern   │ Activates @course-research   │
│ Gateway / workbook     │ Operator console, exploration   │ THIS notebook — httpx only   │
└────────────────────────┴─────────────────────────────────┴──────────────────────────────┘
Gateway pattern: no SDK dependency — just httpx + a running DeerFlow instance.
Swap the BASE_URL to point at a remote server to monitor a production agent.
""")